In [128]:
import pandas as pd
import numpy as np

In [129]:
%store -r 

In [130]:
advance_report =df_dis['Advance Report'].copy()
pipeline = df_dis['Pipeline Report'].copy()
billing = df_dis['Billing Report'].copy()
search_billing = df_dis["Search Billing Sign off"]
bmp_ump = df['BMP_UMP'].copy()
kam_nkam = df['KAM_NKAM'].copy()
master_classification = df['Master_classification'].copy()
Tagging = df['Tagging'].copy()
supercat_wise = df['Supercat wise'].copy()

In [131]:
# print(bmp_ump.columns)
# print("*"*50)
# print(pipeline.columns)
# print("*"*50)
# print(kam_nkam.columns)
# print("*"*50)
# print(billing.columns)
# print("*"*50)
# print(search_billing.columns)
# print("*"*50)
# print(advance_report.columns)
# print("*"*50)
# print(master_classification.columns)
# print("*"*50)
# print(Tagging.columns)
# print("*"*50)
# print(supercat_wise.columns)


In [132]:
kam_nkam = kam_nkam.rename(columns={"Concatenate":"BUxBrand"})

In [133]:
kam_nkam.columns

Index(['seller_id', 'Owner', 'Business Unit', 'Brand-1', 'BUxBrand',
       'ad_Account_id', 'ad_account_name', 'top_seller', 'kam_nkam_tag',
       'channel_tag'],
      dtype='object')

- Function to Convert Object to INT64

In [134]:
def obj2int(series):
    return (
        pd.to_numeric(
            series.astype(str).str.replace(',', '', regex=False), 
            errors='coerce'
        )
        .round(0)
        .astype('Int64')
    )


In [135]:
def flt2int(series):

    if series.dtype == 'object':
        series = series.str.replace(',', '', regex=False)

    return (
        pd.to_numeric(series, errors='coerce') # Invalid strings become NaN
        .round(0)
        .astype('Int64')
    )


# Pipeline Report


- Clean pipeline Data


In [136]:
pipeline["SuperCatagory"] = (
    pipeline["Super Category"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

In [137]:
pipeline['Line Item Net Budget'].sum()

np.float64(2884193093.87)

- Date Check

In [138]:
pipeline['Date Check'] = pipeline['Line Item Start Date'] < StartofMonth

pipeline['Date Check'].value_counts()

Date Check
False    2779
True      360
Name: count, dtype: int64

- Days

In [139]:
diff1 = (PipelineDate - pipeline['Line Item Start Date']).dt.days
diff2 = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days

pipeline['Days'] = np.minimum(diff1, diff2).clip(0)


In [140]:
pipeline['Days'].sum()

np.int64(21940)

- Consumption

In [141]:
duration = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days.clip(lower=1)

pipeline['Consumption'] = np.where(
    pipeline['Date Check'].astype(str) == "True",
    0,
    (pipeline['Line Item Net Budget'] / duration) * pipeline['Days']
)


In [142]:
pipeline['Consumption'].sum()

np.float64(702509913.2161613)

In [143]:
Exception_map = [157770,157771,157773,157774]
pipeline['Exception'] = np.where(pipeline['Line Item Id'].isin(Exception_map), 'Exception', '0')

- Exception

In [144]:
pipeline['Exception'].value_counts()

Exception
0    3139
Name: count, dtype: int64

- Estimates

In [145]:
pipeline['Final_Estimated'] = np.where(pipeline['Exception'] == "Exception", 0, np.where(pipeline['Date Check'] == True, 0, pipeline['Line Item Net Budget']))

In [146]:
pipeline['Final_Estimated'].sum()

np.float64(2246480185.37)

- Rate Card Bifurcation

In [147]:
tag_clean = Tagging.drop_duplicates()
tag_unique = tag_clean.drop_duplicates(subset=['Rate Card'])
rateCard_map = tag_unique.set_index('Rate Card')['Type_RC'].to_dict()
RC_map_data = tag_clean.drop_duplicates(subset=['RC_type_Tag'])
RC_map = dict(zip(RC_map_data['RC_type_Tag'], RC_map_data['RC_Tag']))

pipeline['Rate Card Bifurcation'] = (
    pipeline['Rate Card Dimension']
    .map(rateCard_map)
    .fillna(pipeline['Rate Card Dimension'].map(RC_map))
    .fillna(0)
)


 - Alpha/MP

In [148]:
pipeline['Alpha_MP_Tag'] = np.where(pipeline['SuperCatagory']=="Quich","HL",np.where(pipeline['Line Item Business Type']=="",pipeline['RO Business Type'],pipeline['Line Item Business Type']))

In [149]:
pipeline['New Alpha/MP'] = pipeline['Alpha_MP_Tag'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


- New Supercategory

In [150]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (pipeline['SuperCatagory'] == "Quick DBEFM"),(pipeline['SuperCatagory'] == "Quick Grocery"),
    (pipeline['SuperCatagory'] == "Quick Staples"),(pipeline['SuperCatagory'] == "Quick Healthcare"),
    (pipeline['SuperCatagory'] == "Quick Home"),(pipeline['SuperCatagory'] == "Quick Foods"),
    (pipeline['SuperCatagory'].str.contains("Quick", na=False))
             ]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
              pipeline['RO Brand'].map(map_sheet1).fillna(0)
          ]

default_lookup = pipeline['SuperCatagory'].map(map_mc_b) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_c)) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

pipeline['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [151]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
pipeline['New BU'] = pipeline['New Supercat'].map(bu_mapping).fillna(
    pipeline['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

- RO Start Date

In [152]:

pipeline = pipeline.reset_index(drop=True)

date_mapping = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Start Date']

pipeline['RO Start Date'] = pipeline['Line Item Id'].map(date_mapping).fillna(pd.to_datetime('1900-01-01'))


- Check

In [153]:
last_month_end = LastMonthEnd
pipeline['Check'] = np.where(pipeline['RO End Date'] >= pipeline['RO Start Date'] ,"Yes","No")

- RO Amount

In [154]:

sumifs_q = pipeline.groupby('RO Number')['Line Item Net Budget'].transform('sum')

billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Amount']
xlookup_val = pipeline['Line Item Id'].map(billing_map).fillna(0)

condition = (pipeline['Check'] == "Yes") & (pipeline['RO Start Date'] >= StartofMonth)

calculation = np.divide(xlookup_val, sumifs_q, out=np.zeros_like(xlookup_val), where=sumifs_q!=0) * pipeline['Line Item Net Budget']


pipeline['RO Amount'] = np.where(condition, calculation, 0)


In [155]:
pipeline['RO Amount'].sum()

np.float64(2180817360.42)

In [156]:
pipeline[pipeline['Check'] == "Yes"] ['RO Amount'].sum()

np.float64(2180817360.42)

- Private Brand

In [157]:
pipeline['SCxBrandxAlpha_MP'] = pipeline['New Supercat'].astype(str) + pipeline['RO Brand'].astype(str) + pipeline['New Alpha/MP'].astype(str)
PB_mapping = Tagging.drop_duplicates('SCxBrandxType').set_index('SCxBrandxType')['PrivateBrand']
pipeline['PrivateBrand'] = pipeline['SCxBrandxAlpha_MP'].map(PB_mapping).fillna("0")

- BMP/UMP

In [158]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [159]:
pipeline['BMP_UMP'] = pipeline.apply(BMP_NonBMP, axis=1)

In [160]:
pipeline['BMP_UMP'].value_counts()

BMP_UMP
Non BMP    2953
BMP         186
Name: count, dtype: int64

- KAM/NKAM

In [161]:
pipeline['BUxBrand'] = pipeline['New BU'].astype(str).str.upper().str.strip() + pipeline['RO Brand'].astype(str).str.upper().str.strip()

In [162]:
kam_nkam['BUxBrand'] = kam_nkam['BUxBrand'].astype(str).str.upper().str.strip()
kam_nkam['Brand-1'] = kam_nkam['Brand-1'].astype(str).str.upper().str.strip()
kam_nkam['Owner'] = kam_nkam['Owner'].astype(str).str.upper().str.strip()
pipeline['BUxBrand'] = pipeline['BUxBrand'].astype(str).str.upper().str.strip()
pipeline['Brand'] = pipeline['Brand'].astype(str).str.upper().str.strip()

mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

BUxBrand_KN = pipeline['BUxBrand'].map(mapping_BUXBrand)
Brand_KN = pipeline['Brand'].map(mapping_Brand)

conditions = [ (pipeline['New Supercat'] == "Automobile"), BUxBrand_KN.notna(), Brand_KN.notna() ]

choices = [ "KAM", BUxBrand_KN, Brand_KN ]

pipeline['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [163]:
pipeline['kam_nkam'].value_counts()

kam_nkam
KAM     2116
NKAM    1023
Name: count, dtype: int64

In [164]:
pipeline[pipeline['kam_nkam'] == "KAM"]['Final_Estimated'].sum()

np.float64(1404101924.52)

In [165]:
pipeline.to_excel("Pipeline Report.xlsx",index=False)

# Search Billing Sign Off


- Convert object to integer

In [166]:
# search_billing.columns

In [167]:
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['LineItem Gross Budget'] = obj2int(search_billing['LineItem Gross Budget'])
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['Gross ad spend'] = obj2int(search_billing['Gross ad spend'])
search_billing['LineItem Net Budget'] = obj2int(search_billing['LineItem Net Budget'])

- Clean supercategory

In [168]:
search_billing["SuperCatagory"] = (
    search_billing["Industry.1"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

- New Super Category

In [169]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (search_billing['SuperCatagory'] == "Quick DBEFM"),(search_billing['SuperCatagory'] == "Quick Grocery"),
    (search_billing['SuperCatagory'] == "Quick Staples"),(search_billing['SuperCatagory'] == "Quick Healthcare"),
    (search_billing['SuperCatagory'] == "Quick Home"),(search_billing['SuperCatagory'] == "Quick Foods"),
    (search_billing['SuperCatagory'] == "Quick Beauty & Personal care"),(search_billing['SuperCatagory'] == "Quick Household"),(search_billing['SuperCatagory'] == "Quick Pets"),
    (search_billing['SuperCatagory'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition","Grooming","HouseHold","Pets",
    search_billing['RO Brand'].map(map_sheet1).fillna(0)
]

default_lookup = search_billing['SuperCatagory'].map(map_mc_b) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_c)) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

search_billing['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [170]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
search_billing['New BU'] = search_billing['New Supercat'].map(bu_mapping).fillna(
    search_billing['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

- BMP/UMP


In [171]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [172]:
search_billing['BMP_UMP'] = search_billing.apply(BMP_NonBMP, axis=1)

In [173]:
search_billing["BUxBrand_Cleaned"] = search_billing["New BU"].astype(str).str.upper().str.strip() + search_billing["RO Brand"].astype(str).str.upper().str.strip()

In [174]:
BUxBrand_map =dict(zip(kam_nkam['BUxBrand'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
Brand_map = dict(zip(kam_nkam['Brand-1'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
search_billing['KAM_NKAM_check'] = search_billing['BUxBrand_Cleaned'].map(BUxBrand_map).fillna(search_billing['RO Brand'].astype(str).str.upper().str.strip().map(Brand_map)).fillna("KAM")

In [175]:
search_billing['KAM_NKAM_check'].value_counts()

KAM_NKAM_check
KAM     601
NKAM    139
Name: count, dtype: int64

- BOC/BOL


In [176]:
bol_boc_data = Tagging[['Type_Dimention', 'Check']].drop_duplicates(subset=['Type_Dimention'])
BOL_BOC_map = dict(zip(bol_boc_data['Type_Dimention'], bol_boc_data['Check']))

search_billing['BOC_BOL'] = (
    search_billing['Billing Details']
    .map(BOL_BOC_map)
    .fillna(0)
)


In [177]:
search_billing['BOC_BOL'].value_counts()

BOC_BOL
BOC    335
BOL    291
NO     114
Name: count, dtype: int64

- Consumption

In [178]:
search_billing["Consumption"] = np.where(search_billing['BOC_BOL']=="NO",0,
      search_billing[['LineItem Net Budget','Gross ad spend','Net Billing Number']].min(axis=1))

In [179]:
search_billing["Consumption"].sum()

np.int64(205398735)

- Estimates

In [180]:
gross_spend = search_billing['Gross ad spend'].astype(float)
gross_budget = search_billing['LineItem Gross Budget'].astype(float)

spend_ratio = (gross_spend / gross_budget).fillna(0)
spend_ratio = spend_ratio.replace([np.inf, -np.inf], 0)

conditions = [
    (search_billing['BOC_BOL'] == "NO"),
    (search_billing['BOC_BOL'] == "BOL") & (spend_ratio > SearchBilling_hold)
]

choices = [ 0, search_billing['Net Billing Number'] ]

calc_weighted = (gross_spend / ConsumedDaysSR) * TotalDays
net_budget = search_billing['LineItem Net Budget'].astype(float)
net_billing_num = search_billing['Net Billing Number'].astype(float)
default_case = (
    calc_weighted
    .clip(upper=net_budget)      # Compare weighted vs Net Budget
    #.clip(upper=net_billing_num) # Compare result vs Net Billing Number
)

search_billing['Estimate'] = np.select(conditions, choices, default=default_case)

search_billing['Estimate'] = pd.to_numeric(search_billing['Estimate'], errors='coerce').round(0).astype('Int64')


In [181]:
search_billing['Estimate'].sum()

np.int64(538320686)

- Cons. As a % of Budget

In [182]:
search_billing["Cons. As a % of Budget"] = (search_billing['Consumption']/search_billing['Estimate']).fillna(0)

- Cons. As a % of Net Billing

In [183]:
search_billing["Cons. As a % of Net Billing"] = (search_billing['Consumption']/search_billing['Net Billing Number']).fillna(0)

- Alpha/MP

In [184]:
lookup_map = Tagging.set_index('Supercat')['Alpha/MP'].to_dict()

In [185]:
search_billing = search_billing.reset_index(drop=True)

if not isinstance(lookup_map, dict):
    lookup_map = dict(lookup_map)

cond_quick = search_billing['SuperCatagory'].str.contains("Quick", na=False, case=False)
cond_lookup = search_billing['New Supercat'].isin(lookup_map.keys())

choices = [
    "HL", 
    search_billing['New Supercat'].apply(lambda x: lookup_map.get(x, np.nan))
]

is_empty_biz = search_billing['Line Item Business Type'].isna() | (search_billing['Line Item Business Type'].astype(str).str.strip() == "")
is_bmp = search_billing['Line Item Business Type'] == "BMP"

default_logic = np.select(
    [is_empty_biz, is_bmp],
    [search_billing['RO Business Type'], "MP"],
    default=search_billing['Line Item Business Type']
)

search_billing['Alpha_MP'] = np.select(
    [cond_quick.values, cond_lookup.values],
    [np.broadcast_to("HL", len(search_billing)), choices[1].values],
    default=default_logic
)


In [186]:
search_billing['Alpha_MP'].value_counts()

Alpha_MP
MP       440
Alpha    253
HL        45
3P         2
Name: count, dtype: int64

In [187]:
search_billing[(search_billing['Alpha_MP'] == "HL") & (search_billing['New BU'] == 0)]


,Row #,Ext RO Number Flag,External RO number,Billing as per the Internal RO Number?,Internal RO number,RO Business Type,Line Item Business Type,Advertiser,Sales Manager,Line Item Name,...,New BU,BMP_UMP,BUxBrand_Cleaned,KAM_NKAM_check,BOC_BOL,Consumption,Estimate,Cons. As a % of Budget,Cons. As a % of Net Billing,Alpha_MP


- KAM/NKAM

In [188]:
search_billing["BUxBrand"] = search_billing["New BU"].astype(str) + search_billing["RO Brand"].astype(str)

In [189]:
kam_nkam.columns

Index(['seller_id', 'Owner', 'Business Unit', 'Brand-1', 'BUxBrand',
       'ad_Account_id', 'ad_account_name', 'top_seller', 'kam_nkam_tag',
       'channel_tag'],
      dtype='object')

In [190]:
mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

SBUxBrand_KN = search_billing['BUxBrand'].map(mapping_BUXBrand)
SBrand_KN = search_billing['RO Brand'].map(mapping_Brand)

conditions = [ (search_billing['New Supercat'] == "Automobile"), SBUxBrand_KN.notna(), SBrand_KN.notna() ]

choices = [ "KAM", SBUxBrand_KN, SBrand_KN ]

search_billing['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [191]:
search_billing['kam_nkam'].value_counts()

kam_nkam
KAM     715
NKAM     25
Name: count, dtype: int64

In [192]:
search_billing[search_billing['SuperCatagory'] == "Automobile"]['New Supercat'].value_counts()

New Supercat
Automobile    13
Name: count, dtype: int64

In [193]:
search_billing_col = [ "Row #", "Ext RO Number Flag", "External RO number",
    "Billing as per the Internal RO Number?","Internal RO number","RO Business Type",
    "Line Item Business Type", "Advertiser", "Sales Manager", "Line Item Name",
    "Line Item Id", "Industry", "Inventory Slot Wise", "Start Date", "End Date",
    "Cost Type","Original Rate","Negotiated Rate","LineItem Gross Budget",
    "LineItem Net Budget","LineItem Discount","RO Link","RO Supporting Document - External",
    "Internal Additional Document","Folder Link","Campaign Id","Remaining Budget",
    "Actual Revenue","Gross Actual Revenue","Net Actual Revenue","Final Billing Revenue",
    "Billed Revenue","Over/Under Delivery","POE","POE Type","POC Name","Agency/Direct",
    "1P/3P","Industry.1","RO Brand","RO Brand Text","RO Brand Grouping","Mall Eligibility",
    "RO Amount","RO Start Date","RO End Date","RO Period","RO Amount Without Tax",
    "Tax","RO Amount With Tax","Amount to be billed by FKN to brand","GST","CEF Number",
    "Pan Card Number","Signed RO","CEF","Pan Card Copy","Payment Terms","Folder Link.1",
    "Invoice Status","Supporting Doc Link","LI External Supporting Doc Link","BU",
    "Gross ad spend","Gross Billing Number","Net Billing Number","Billing Details",
    "Add account id","BOC_BOL", "Consumption","Estimate", "Cons. As a % of Budget",
    "Cons. As a % of Net Billing","Alpha_MP","New Supercat","New BU","BMP_UMP", "kam_nkam"
]


In [194]:
search_billing[search_billing_col].to_excel("Search Billing.xlsx",index=False)

# Advance Report


In [195]:
advance = advance_report.copy()

In [196]:
billing['Line Item Id'] = flt2int(billing['Line Item Id'])

In [197]:
billing['Line Item Id'].dtype

Int64Dtype()

- Line Item Id


In [198]:
advance.loc[:, 'Line Item Id'] = (
    advance["Name"].str.extract(r"(\d+)")
    .fillna(0)
    .astype(int)
)

In [199]:
advance['Line Item Id'].count()

np.int64(2007)

In [200]:
advance[advance['Line Item Id'] == 0]

,ID,Name,Total Budget,Daily Budget,Remaining Days,Discount,Remaining Budget,Impressions,Views,Total Spend,...,Orders (View Attribution),Conversion Rate,Revenue (Click Attribution),Revenue (View Attribution),Orders,Revenue,ROI,Leads,Lead Rate,Line Item Id
211,3RAXOM59NGYL,NaN,0.0,NaN,NaN,0.0,0.0,0,1,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
248,4FLGKXNMIII9,NaN,0.0,NaN,NaN,0.0,0.0,0,0,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
962,H86TMX9D71FP,NaN,0.0,NaN,NaN,0.0,0.0,74433,5912,0.0,...,18,26.0274,84990.0,301562.0,19,386552.0,0.0000,0,0.0,0
992,HSB7H4BEUMZH,NaN,0.0,NaN,NaN,0.0,0.0,249730,4934,0.0,...,4,7.2727,0.0,75496.0,4,75496.0,0.0000,0,0.0,0
1095,JLAQG7L20SOR,Drawer,100.0,NaN,0.0,100.0,100.0,0,0,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
1167,KT706DTQHMO1,NaN,0.0,NaN,NaN,0.0,0.0,0,0,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
1514,QXQW6FFTO5T5,NaN,0.0,NaN,NaN,0.0,0.0,0,0,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
1817,W3K97OB5M35W,NaN,0.0,NaN,NaN,0.0,0.0,10,33,0.0,...,0,NaN,0.0,0.0,0,0.0,0.0000,0,NaN,0
1853,WUQ9ZYTXKF4B,Test,500.0,NaN,0.0,52.2,-30464.0,691013,61928,30964.0,...,5,0.7384,65298.0,52246.0,7,117544.0,3.7962,0,0.0,0


- Supercat

In [201]:
map_p1 = pipeline.set_index('Line Item Name')['SuperCatagory'].to_dict()
map_p2 = pipeline.set_index('Line Item Id')['SuperCatagory'].to_dict()
map_b3 = billing.set_index('Line Item Id')['Industry.1'].to_dict()

advance['Supercat'] = advance['Name'].map(map_p1)

advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_p2))
advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_b3))

advance['Supercat'] = advance['Supercat'].fillna(0)


In [202]:
advance['Supercat'].value_counts()

Supercat
Grooming             268
LaptopAndDesktop     131
IOT                  105
Mobile                98
Quick Foods           98
                    ... 
Quick Electronics      2
WomenEthnic            1
FashionWearables       1
HouseHoldSupplies      1
Sellback               1
Name: count, Length: 66, dtype: int64

- Brand


In [203]:
map_m_to_b = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['RO Brand']
map_l_to_b = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Brand']

advance['Brand'] = (
    advance['Name'].map(map_m_to_b)
    .fillna(advance['Line Item Id'].map(map_l_to_b))
    .fillna(0)
)


In [204]:
advance['Brand'] = advance['Brand'].str.upper()

In [205]:
Tagging['Brand Name'] = Tagging['Brand Name'].str.upper()

- New SuperCat

In [206]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_ba = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_ca = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df_a = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_ea = temp_df.index.to_series()

conditions = [
    (advance['Supercat'] == "Quick DBEFM"),(advance['Supercat'] == "Quick Grocery"),
    (advance['Supercat'] == "Quick Staples"),(advance['Supercat'] == "Quick Healthcare"),
    (advance['Supercat'] == "Quick Home"),(advance['Supercat'] == "Quick Foods"),
    (advance['Supercat'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition", 
    advance['Brand'].map(map_sheet1).fillna(0)
]

default_lookup = advance['Supercat'].map(map_mc_b) \
                 .fillna(advance['Supercat'].map(map_mc_c)) \
                 .fillna(advance['Supercat'].map(map_mc_e)) \
                 .fillna(0)

advance['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU


In [207]:
master_map = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')['BU']

advance['New BU'] = advance['New Supercat'].map(master_map).fillna(advance['New Supercat'].replace({"Staples": "Grocery"}))


- Alpha/MP

In [208]:
map_sheet1_no = (
    Tagging.drop_duplicates('Supercat')
    .set_index('Supercat')['Alpha/MP']
    .to_dict()
)

map_pipe_mae = (
    pipeline.drop_duplicates('Line Item Name')
    .set_index('Line Item Name')['RO Business Type']
    .to_dict()
)

map_pipe_lae = (
    pipeline.drop_duplicates('Line Item Id')
    .set_index('Line Item Id')['RO Business Type']
    .to_dict()
)

is_quick = advance['Supercat'].str.contains("Quick", case=False, na=False)

is_in_sheet1 = advance['New Supercat'].isin(Tagging['Supercat'])

pipeline_lookup = (
    advance['Name'].map(map_pipe_mae)
    .fillna(advance['Line Item Id'].map(map_pipe_lae))
    .fillna(0)
)

conditions = [
    is_quick,      
    is_in_sheet1   
]

choices = [
    "HL",                                         
    advance['New Supercat'].map(map_sheet1_no)    
]

advance['Alpha_MP'] = np.select(conditions, choices, default=pipeline_lookup)


In [209]:
advance['Alpha_MP'].value_counts()

Alpha_MP
Alpha    1411
HL        311
MP        156
0          62
BMP        43
3P         24
Name: count, dtype: int64

- New Alpha/MP

In [210]:
advance['New Alpha/MP'] = advance['Alpha_MP'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


In [211]:
advance['New Alpha/MP'].value_counts()

New Alpha/MP
Alpha    1411
HL        311
MP        199
0          62
3P         24
Name: count, dtype: int64

- Line Item Net Budget

In [212]:
map_p_m_q = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Net Budget']
map_p_l_q = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Net Budget']
map_b_k_t = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['LineItem Net Budget']

advance['Line Item Net Budget'] = (
    advance['Name'].map(map_p_m_q)
    .fillna(advance['Line Item Id'].map(map_p_l_q))
    .fillna(advance['Line Item Id'].map(map_b_k_t))
    .fillna(0)
)


In [213]:
advance['Line Item Net Budget'].sum()

np.float64(1716024821.0900002)

Check (Supercat Match)

In [214]:
advance['Check'] = advance['New Supercat'].isin(supercat_wise['Super Categories'])


In [215]:
advance['Check'].value_counts()

Check
True     1911
False      96
Name: count, dtype: int64

- Start Date


In [216]:
map_m_to_o = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Start Date']
map_l_to_o = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Start Date']

advance['Start_Date'] = (
    advance['Name'].map(map_m_to_o)
    .fillna(advance['Line Item Id'].map(map_l_to_o))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [217]:
advance['Start_Date'].head()

0   1900-01-01
1   2026-05-09
2   2026-05-09
3   2026-05-01
4   2026-05-01
Name: Start_Date, dtype: datetime64[ns]

- End Date

In [218]:
map_m_to_p = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item End Date']
map_l_to_p = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item End Date']

advance['End_Date'] = (
    advance['Name'].map(map_m_to_p)
    .fillna(advance['Line Item Id'].map(map_l_to_p))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [219]:
advance['End_Date'].head()

0   1900-01-01
1   2026-05-15
2   2026-05-15
3   2026-05-31
4   2026-05-20
Name: End_Date, dtype: datetime64[ns]

- CPD/CMP

In [220]:
map_m_to_x = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Rate Card']
map_l_to_x = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Rate Card']

advance['CPD_CPM'] = (
    advance['Name'].map(map_m_to_x)
    .fillna(advance['Line Item Id'].map(map_l_to_x))
    .fillna(0)
)

In [221]:
advance['CPD_CPM'].value_counts()

CPD_CPM
vCPM                                     1074
vCPM Rate Card                            462
Less than 100 SOV                         328
0                                          62
100 SOV - 1 Brand 1 RO - CPD - EBD         43
100 SOV - 1 Brand 1 RO CPD-Normal BAU      20
100 SOV - CPD                              13
CPM                                         4
less than 100 SOV                           1
Name: count, dtype: int64

- BOL/BOC

In [222]:
bol_list = ["LESS THAN 100 SOV", "CPD", "CPT", "RESERVATION BUY", "100 SOV - CPD", "100 SOV" ]
advance['BOL/BOC'] = np.where(advance['CPD_CPM'].str.upper().str.strip().isin(bol_list), "BOL", "BOC")

In [223]:
advance['BOL/BOC'].value_counts()

BOL/BOC
BOC    1665
BOL     342
Name: count, dtype: int64

- Date_Check


In [224]:
advance['Date_Check'] = np.where(advance['BOL/BOC'] == 'BOC', "False", advance['Start_Date'] < StartofMonth)

In [225]:
advance['Date_Check'].value_counts()

Date_Check
False    1972
True       35
Name: count, dtype: int64

- Date of Report

In [226]:
advance['Date of Report'] =  AdvanceDate

- No of Days


In [227]:
advance['NoofDays'] = np.maximum(
    np.minimum(
        (advance["Date of Report"] - advance["Start_Date"]).dt.days,
        (advance["End_Date"] - advance["Start_Date"]).dt.days
    ),
    1
)

In [228]:
advance['NoofDays'].sum()

np.int64(10637)

- Final Consumption Numbers

In [229]:
duration = (advance['End_Date'] - advance['Start_Date']).dt.days.clip(lower=1)

conditions = [
    (advance['Date_Check'] == "True"),
    (advance['BOL/BOC'] == "BOL")
]

choices = [
    0,
    (advance['Line Item Net Budget'] / duration) * advance['NoofDays'],
]

advance['Final Consumption Numbers'] = np.select(
    conditions,
    choices,
    default=np.minimum(advance['Net Spend'], advance['Line Item Net Budget'])
)


In [230]:
advance['Final Consumption Numbers'].sum()

np.float64(683339656.5042561)

- Final Estimates

In [231]:
advance["Final Estimates"] = np.where(
    (advance["Date_Check"] == True) | (advance["Final Consumption Numbers"] == 0),
    0,
    np.where(
        advance["BOL/BOC"] == "BOL",
        advance["Line Item Net Budget"],
        np.minimum(
            ((advance["Net Spend"] * TotalDays) / ConsumedDaysAR),
            advance["Line Item Net Budget"],
        ),
    ),
)




In [232]:
advance['Final Estimates'].sum()

np.float64(1540014878.1596088)

- Private brand


In [233]:
advance['LookupKey'] = advance['New Supercat'].astype(str) + advance['Brand'].astype(str) + advance['New Alpha/MP'].astype(str)
Tagging['LookupKey'] = Tagging['Supercat_PB'].astype(str) + Tagging['Brand_PB'].astype(str) + Tagging['Type_PB'].astype(str)

tagging_map = Tagging.drop_duplicates('LookupKey').set_index('LookupKey')['PrivateBrand']

advance['Private Brand'] = advance['LookupKey'].map(tagging_map)


In [234]:
advance['Private Brand'].value_counts()

Series([], Name: count, dtype: int64)

- KAM/NKAM


In [235]:
map_BUxBrand = kam_nkam.drop_duplicates('BUxBrand').set_index('BUxBrand')['Owner']
map_Brand = kam_nkam.drop_duplicates('Brand-1').set_index('Brand-1')['Owner']
advance['BUxBrand'] = advance['New BU'].astype(str) + advance['Brand'].astype(str)

condition = (advance['New Supercat'] == "Automobile")

BUxBrand_mapped = advance['BUxBrand'].map(map_BUxBrand).fillna("KAM")

final_map = advance['Brand'].map(map_Brand).fillna(BUxBrand_mapped)

conditions = [condition]
choices = ["KAM"]

advance['KAM/NKAM'] = np.select(conditions, choices, default=final_map)


In [236]:
advance[advance['KAM/NKAM'] == "KAM"]['Final Estimates'].sum()

np.float64(720371287.8606272)

- BMP/UMP

In [237]:
advance['BMP/UMP'] = np.where(advance['New Supercat'] == "Automobile", "BMP",np.where(advance['Alpha_MP'] == "BMP", "BMP","NonBMP"))

In [238]:
advance['BMP/UMP'].value_counts()

BMP/UMP
NonBMP    1910
BMP         97
Name: count, dtype: int64

- Internal RO Number

In [239]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal RO number']

advance['Internal RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)


- External RO NO

In [240]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['External RO number']

advance['External RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Advertiser

In [241]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Advertiser']

advance['Advertiser'] = advance['Line Item Id'].map(billing_map).fillna(0)

- PAN Number

In [242]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Pan Card Number']

advance['PAN Number'] = advance['Line Item Id'].map(billing_map).fillna(0)

- GSTIN

In [243]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['GST']

advance['GSTIN'] = advance['Line Item Id'].map(billing_map).fillna(0)

- RO Link

In [244]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Link']

advance['RO Link'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Internal Additional Document

In [245]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal Additional Document']

advance['Additional Document'] = advance['Line Item Id'].map(billing_map).fillna(0)

- loaded vs Estimates

In [246]:
advance['Loaded Vs Estimates'] = advance['Line Item Net Budget'] - advance['Final Estimates']

In [247]:
advance['Loaded Vs Estimates'].sum()

np.float64(176009942.93039092)

- Net Spend Vs Estimates

In [254]:
advance['NetSpend vs Estimates'] = advance['Net Spend'] - advance['Final Estimates']

In [255]:
advance['NetSpend vs Estimates'].sum()

np.float64(-1156188611.172309)

In [250]:
ad_grp = advance.groupby(['New BU','New Alpha/MP'])['Final Consumption Numbers'].sum()

In [251]:
# print(ad_grp)

In [257]:
advance.to_excel("Advance Report.xlsx",index=False)

In [253]:
%store pipeline search_billing advance

Stored 'pipeline' (DataFrame)
Stored 'search_billing' (DataFrame)
Stored 'advance' (DataFrame)
